In [1]:
from pathlib import Path

from functools import lru_cache
from lightning import pytorch as pl
import numpy as np
import pandas as pd
from rdkit import Chem
from torch.utils.data import DataLoader

from chemprop.data import (
    MoleculeDatapoint,
    MoleculeDataset,
    make_split_indices,
    split_data_by_indices,
)
from chemprop.nn.agg import NormAggregation
from chemprop.nn.message_passing import BondMessagePassing
from chemprop.nn.predictors import RegressionFFN
from chemprop.nn.transforms import UnscaleTransform

In [2]:
INPUT_PATH = Path("tests/test_data.csv")
INCHI_COLS = ["inchi_solute", "inchi_solvent1", "inchi_solvent2"]
FRAC_COL = "frac_solvent1"
TARGET_COLS = ["Gsolv (kcal/mol)"]


@lru_cache(maxsize=None)
def inchi_to_mol(inchi: str):
    if inchi is None or (isinstance(inchi, float) and np.isnan(inchi)):
        return None
    return Chem.MolFromInchi(inchi)


df = pd.read_csv(INPUT_PATH)

ys = df[TARGET_COLS].to_numpy(dtype=float)
mol_solute = df[INCHI_COLS[0]].map(inchi_to_mol)

# Account for some mono-solvent rows here to make creating the datapoints easier, see below.
fracs = df[FRAC_COL].to_numpy(dtype=float)
inchis_solvent1 = df[INCHI_COLS[1]]
inchis_solvent2 = df[INCHI_COLS[2]]

only_solvent2 = fracs == 0.0 # Swap solvent 1 and 2 when no solvent 1
only_one_solvent = only_solvent2 | (fracs == 1.0) # Make solvent 2 None when only one solvent

inchis_solvent1 = inchis_solvent1.where(~only_solvent2, inchis_solvent2)
inchis_solvent2 = inchis_solvent2.where(~only_one_solvent, None)
fracs = np.where(only_solvent2, 1.0, fracs)

mol_solvent1 = inchis_solvent1.map(inchi_to_mol)
mol_solvent2 = inchis_solvent2.map(inchi_to_mol)

print(f"Loaded {len(df)} rows")

Loaded 50 rows


### Introducing *Components* as the molecules that are part of a mixture (order invariant)

A `ComponentDatapoint` is like a `MoleculeDatapoint` but adds the attribute `w_fp`, which is a pre-determined weight of the learned fingerprint of the molecule when combining the fingerprints of components in the mixture. (This could be the component's mole fraction, for example.)

A `ComponentDataset` uses `MoleculeOrNoneMolGraphFeaturizer` as the default featurizer, which is the same as `SimpleMoleculeMolGraphFeaturizer`, but allows for the input to be `None` (returns `None`).

### Introducing *Mixtures* as objects that represent molecules and their interactions 

A `MixtureDatapoint` is like a `MoleculeDatapoint` but uses molecules instead of atoms as nodes of the graph and interactions instead of bonds as the edges (a `MixtureGraph`). This is required if using the optional mixture message passing classes, see below.

We implement the ``SimpleMixtureGraphFeaturizer`` for ``MixtureGraph`` objects that takes in the molecules of a mixture, constructs a mixture graph, and adds descriptors for intermolecular interactions to the edge features (currently only considering hydrogen bonding). The node features are filled in with the learned fingerprints of the molecules after atom-to-molecule aggregation. In the future we may add a molecule featurizer to supplement this vector with calculable molecule features.

In [3]:
from chemprop_contrib.mixtures.data.datapoints import ComponentDatapoint, MixtureDatapoint

all_data = [[MoleculeDatapoint(mol, y) for mol, y in zip(mol_solute, ys)]]
all_data += [[ComponentDatapoint(mol, w_fp=f) for mol, f in zip(mol_solvent1, fracs)]]
all_data += [[ComponentDatapoint(mol, w_fp=f) for mol, f in zip(mol_solvent2, fracs)]]
all_data += [
    [
        MixtureDatapoint([mol for mol in mols if mol is not None])
        for mols in zip(mol_solute, mol_solvent1, mol_solvent2)
    ]
]

In [4]:
train_idx, val_idx, test_idx = make_split_indices(range(len(all_data[0])))
train_data, val_data, test_data = split_data_by_indices(all_data, train_idx, val_idx, test_idx)
train_data, val_data, test_data = train_data[0], val_data[0], test_data[0]

The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)


*Note*: MixtureGraphDataset is assumed to be the final dataset.

In [5]:
from chemprop_contrib.mixtures.data.datasets import (
    MixtureDataset,
    ComponentDataset,
    MixtureGraphDataset,
)


def make_dataset(datapoints):
    datasets = [
        MoleculeDataset(datapoints[0]),
        ComponentDataset(datapoints[1]),
        ComponentDataset(datapoints[2]),
        MixtureGraphDataset(datapoints[3]),
    ]
    for dataset in datasets:
        dataset.cache = True
    return MixtureDataset(datasets)


train_mcdset = make_dataset(train_data)
val_mcdset = make_dataset(val_data)
test_mcdset = make_dataset(test_data)

scaler = train_mcdset.normalize_targets()
scaler = val_mcdset.normalize_targets(scaler)

In [6]:
from chemprop_contrib.mixtures.data.collate import collate_mixture

train_loader = DataLoader(train_mcdset, batch_size=64, shuffle=True, collate_fn=collate_mixture)
val_loader = DataLoader(val_mcdset, batch_size=64, shuffle=False, collate_fn=collate_mixture)
test_loader = DataLoader(test_mcdset, batch_size=64, shuffle=False, collate_fn=collate_mixture)

### Message passing with multiple components from different ``groups``

`MixtureMulticomponentMessagePassing` is like `MulticomponentMessagePassing` but adds the argument `groups`.

The groups provide the indices for the respective molecular/component-wise datasets. For example, ``groups = [[0], [1, 2, 3]]``, where the first group corresponds to solute (1 molecule) and the second group corresponds to solvents (3 molecules). If all molecules/components are part of a mixture, one single group should be used, i.e., ``groups = [[0, 1, 2, 3]]``.

All components within each group use the same message passing block. Also, if ``shared = True``, all groups share a message passing block.

In [7]:
from chemprop_contrib.mixtures.nn.message_passing import MixtureMulticomponentMessagePassing

GROUPS = [[0], [1, 2]]  # solute, solvent1 + solvent2
mcmp = MixtureMulticomponentMessagePassing(
    blocks=[
        BondMessagePassing(),  # solute
        BondMessagePassing(),  # shared by solvents
    ],
    groups=GROUPS,
    shared=False,
)

After message passing each component separately, the directed edges are aggregated into a learned fingerprint for each component, as is typical in `chemprop`. 

In [8]:
graph_agg = NormAggregation()

### Mixture-level message passing

Once we have the learned fingerprints for each component, we can optionally perform message passing between all molecules in the datapoint by embedding them in a graph to allow components to inform each other's representations, analogous to intermolecular interactions. (Future work is to allow for message passing between only components in a given mixture.) We provide two types of message passing on the mixture level:
* On-the-fly: a fully-connected mixture graph is constructed during the forward pass of the model, without any edge features (no molecular interaction descriptors are calculated). That is, we just enable passing information between the learned molecular fingerprints.
* MixtureGraph: a ``MixtureGraphDataset`` (see above) must be generated and passed to the model. This dataset uses a featurizer that calculates properties of the interactions as edge features (currently only hydrogen bonding.) We  describe two classes for message passing these graphs: ``InteractionMessagePassing``, like ``BondMessagePassing``, (edge-centered), and ``MolecularMessagePassing``, like ``AtomMessagePassing`` (node-centered). 

This can also be left as `None` for no message passing between components.

In [9]:
from chemprop_contrib.mixtures.featurizers import SimpleMixtureGraphFeaturizer
from chemprop_contrib.mixtures.nn.message_passing import (
    MixtureMessagePassing,
    InteractionMessagePassing,
    MolecularMessagePassing,
)

DEFAULT_MOL_FDIM, DEFAULT_INTERACTION_FDIM = SimpleMixtureGraphFeaturizer().shape
mixmp_class = InteractionMessagePassing  # or MolecularMessagePassing

mixmp = mixmp_class(
    # The vertex (node) dimension must match the output of the molecule message passing
    # and all molecule message passing blocks must have the same output dimension.
    d_v=mcmp.blocks[0].output_dim + DEFAULT_MOL_FDIM,
    d_e=DEFAULT_INTERACTION_FDIM,
)

# MixtureMessagePassing doesn't use edge features
# mixmp = MixtureMessagePassing(d_v=mcmp.blocks[0].output_dim)

### ``MixtureAggregation``: From molecular to mixture representations

Once we have the final component representations, we combine them into a single vector for the feed-forward neural network using a mixture aggregation.

We currently include: 
* ``ConcatAggregation``: Simply concatenating all molecular fingerprints and the individual compositions.
* ``WeightedSumAggregation``: Groups-wise sum of molecular fingerprints multiplied by their individual compositions.
* ``DeepsetsAggregation``: Can be seen as an extension of ``WeightedSumAggregation``, whereas the individual molecular fingerprints multiplied by the compositions pass a _local_ MLP before being summed and then the group-wise sums pass a _global_ MLP.
* ``AttentiveAggregation``: Groups-wise attention layer applied to molecular fingerprints multiplied by their individual compositions (meaning that the weighting of the individual fingerprints is adjusted by attention logits).
* ``Set2SetAggregation``: Recurrent architecture based on LSTMs that aggregate group-wise molecular fingerprints multiplied by their individual composition into a mixture representation.

*Note*: All mixture aggregations operate group-wise and then concatenate the group-based molecular/mixture representations

In [10]:
from chemprop_contrib.mixtures.nn.agg import (
    AttentiveAggregation,
    ConcatAggregation,
    DeepsetsAggregation,
    Set2SetAggregation,
    WeightedSumAggregation,
)

mixture_agg_classes = {
    "weightedsum": WeightedSumAggregation,
    "cat": ConcatAggregation,
    "deepsets": DeepsetsAggregation,
    "attentive": AttentiveAggregation,
    "set2set": Set2SetAggregation,
}

if mixmp is None:
    fp_dims = mcmp.output_dims
else:
    fp_dims = [mixmp.output_dim] * len(mcmp.blocks)

mixagg = mixture_agg_classes["weightedsum"](
    graph_agg=graph_agg,
    groups=GROUPS,
    fp_dims=fp_dims,
    mixmp=mixmp,
)

In [11]:
from chemprop_contrib.mixtures.models import MixtureMPNN

output_transform = UnscaleTransform.from_standard_scaler(scaler)
ffn = RegressionFFN(
    input_dim=mixagg.output_dim,
    output_transform=output_transform,
)

mcmpnn = MixtureMPNN(mcmp, mixagg, ffn)
mcmpnn

MixtureMPNN(
  (message_passing): MixtureMulticomponentMessagePassing(
    (blocks): ModuleList(
      (0-2): 3 x BondMessagePassing(
        (W_i): Linear(in_features=86, out_features=300, bias=False)
        (W_h): Linear(in_features=300, out_features=300, bias=False)
        (W_o): Linear(in_features=372, out_features=300, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (tau): ReLU()
        (V_d_transform): Identity()
        (graph_transform): Identity()
      )
    )
  )
  (agg): WeightedSumAggregation(
    (graph_agg): NormAggregation()
    (mixmp): InteractionMessagePassing(
      (W_i): Linear(in_features=305, out_features=300, bias=False)
      (W_h): Linear(in_features=300, out_features=300, bias=False)
      (W_o): Linear(in_features=600, out_features=300, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
      (tau): ReLU()
      (V_d_transform): Identity()
      (graph_transform): Identity()
    )
  )
  (bn): Identity()
  (predictor): Regressio

### Training

In [12]:
trainer = pl.Trainer(
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=True,
    accelerator="auto",
    devices=1,
    max_epochs=20,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [13]:
trainer.fit(mcmpnn, train_loader, val_loader)

Loading `train_dataloader` to estimate number of stepping batches.
/home/knathan/anaconda3/envs/chemprop_contrib/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

  | Name            | Type                                | Params | Mode 
--------------------------------------------------------------------------------
0 | message_passing | MixtureMulticomponentMessagePassing | 455 K  | train
1 | agg             | WeightedSumAggregation              | 361 K  | train
2 | bn              | Identity                            | 0      | train
3 | predictor       | RegressionFFN                       | 180 K  | train
4 | X_d_transform   | Identity                            | 0      | train
5 | metrics         | ModuleList                          | 0   

Sanity Checking DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]

/home/knathan/anaconda3/envs/chemprop_contrib/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Epoch 19: 100%|██████████| 1/1 [00:00<00:00,  9.86it/s, train_loss_step=0.0667, val_loss=0.0465, train_loss_epoch=0.0667]

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 19: 100%|██████████| 1/1 [00:00<00:00,  9.67it/s, train_loss_step=0.0667, val_loss=0.0465, train_loss_epoch=0.0667]


In [14]:
results = trainer.test(mcmpnn, test_loader)

/home/knathan/anaconda3/envs/chemprop_contrib/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 88.46it/s] 


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/mse          │     6.212258338928223     │
└───────────────────────────┴───────────────────────────┘